In [ ]:
import os
from pathlib import Path

import contextily as ctx
import dask_geopandas as dg
import geopandas as gpd
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from matplotlib import colors

### Find tiles

In [ ]:
swot_dir = '/nas/cee-ice/data/SWOT/PIXC_filtered_vD/'
swot_dir = Path(swot_dir)

# virgin_tile = '_496_090L_'
virgin_tile = '_190_089R_'

pixc_files = [
    str(p) for p in swot_dir.rglob('*') 
    if p.is_file() and virgin_tile in p.name
]

In [ ]:
# Make dict for legend labels
flags = gdf_PIXC.classification.unique()
codes = {idx:k for idx, k in enumerate(flags, start=2)}

In [ ]:
palette = {2: 'darkolivegreen', 3: 'cornflowerblue', 4: 'blue', 5: 'hotpink',
           6: 'darkkhaki', 7: 'springgreen'}

In [ ]:
# bounds = [240007, 4085768, 245007, 4092438]
# bounds = [283318, 4110711, 287318, 4115711]
bounds = [274101, 4107524, 278101, 4112524]
# bounds = [261045, 4098043, 265045, 4103043]
buffer_c = buffer.clip(mask=bounds)

for pixc_file in pixc_files[0:1]:
    test_pixc = pd.read_parquet(os.path.join(swot_dir, pixc_file))

    test_pixc = gpd.GeoDataFrame(
    test_pixc, 
    geometry=gpd.points_from_xy(test_pixc.longitude_WGS84, test_pixc.latitude_WGS84),
    crs="EPSG:4326"
    )

    test_pixc = test_pixc.to_crs('EPSG:32612')
    granule_name = pixc_file[-60:-39]

    test_c = test_pixc.clip(mask=bounds)
    
    nums = list(np.unique(test_pixc.classification).astype('int'))
    labels = [codes[x] for x in nums]
    
    # Make cmap
    cmap = colors.ListedColormap([palette[x] for x in nums])
    
    fig, ax = plt.subplots(figsize=(5, 6.667))
    buffer_c.plot(ax=ax, alpha=0.5, color='deeppink')
    test_c.plot(column='classification', cmap=cmap, ax=ax, s=0.5)

    # ctx.add_basemap(ax, crs=test_pixc.crs, source=ctx.providers.CartoDB.Positron, attribution='')
    # ax.text(0.99, 0.01, ctx.providers.CartoDB.Positron.attribution, 
    #         transform=ax.transAxes, 
    #         ha='right', va='bottom', 
    #         fontsize=7, 
    #         path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2, foreground="w")])

    plt.xlim(bounds[0], bounds[2])
    plt.ylim(bounds[1], bounds[3])
    
    plt.title(granule_name)
    plt.show()

### Functions

In [ ]:
def read_and_prep_pixc(granule, epsg_code):
    # Find the right file
    swot_dir = Path('/nas/cee-ice/data/SWOT/PIXC_filtered_vD/')
    pixc_file = next(swot_dir.rglob(f"*{granule}*"), None)

    # Read in a prep
    gdf_PIXC = pd.read_parquet(pixc_file)
    
    gdf_PIXC = gpd.GeoDataFrame(
        gdf_PIXC, 
        geometry=gpd.points_from_xy(gdf_PIXC.longitude_WGS84,
                                    gdf_PIXC.latitude_WGS84),
        crs="EPSG:4326"
    )

    # Convert CRS to WGS 84 / UTM zone XXX
    gdf_PIXC = gdf_PIXC.to_crs(epsg=epsg_code)

    return gdf_PIXC

In [ ]:
def read_and_prep_nhd_buffer(huc4, epsg_code, extra):
    huc2 = huc4[0:2]

    if extra ==True:
        data_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered_extra/HUC2_{huc2}/'
        file_path = os.path.join(data_dir,
                                 f'NHDPLUS_H_{huc4}_HU4_GDB_prepped_buffered_extra_mean.parquet')
        
        buffer = gpd.read_parquet(file_path)
        # Convert CRS to WGS 84 / UTM zone XXX
        buffer = buffer.to_crs(epsg=epsg_code)
        
    elif extra == False:
        data_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered/HUC2_{huc2}/'
        file_path = os.path.join(data_dir,
                                 f'NHDPLUS_H_{huc4}_HU4_GDB_prepped_buffered_mean.parquet')
        
        buffer = gpd.read_parquet(file_path)
        # Convert CRS to WGS 84 / UTM zone XXX
        buffer = buffer.to_crs(epsg=epsg_code)

    return buffer

In [ ]:
save_path = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/figures/'

### Virgin River

In [ ]:
granule = '_011_496_090L_'
# granules = '_XXX_190_089R_'
epsg_code = '32612' # WGS 84 / UTM zone 12N
huc4 = '1501'
# bounds = [240007, 4085768, 245007, 4092438]
bounds = [241007, 4087603, 244007, 4090603]

# bounds = [240000, 4085500, 245000, 4092500]
# bounds = [283318, 4110711, 287318, 4115711]

In [ ]:
gdf_PIXC = read_and_prep_pixc(granule=granule, epsg_code=epsg_code)

# Read in the buffered reaches (for plotting)
# and the buffered reaches with a bit extra (for clipping pixels to show corridor)
buffer = read_and_prep_nhd_buffer(huc4=huc4, epsg_code=epsg_code, extra=False)
buffer_extra = read_and_prep_nhd_buffer(huc4=huc4, epsg_code=epsg_code, extra=True)

# Clip to bounds
gdf_PIXC_clip = gdf_PIXC.clip(mask=bounds)
gdf_PIXC_buffer = gdf_PIXC_clip.sjoin(buffer_extra)
buffer_clip = buffer.clip(mask=bounds)

print(buffer_clip[buffer_clip['WidthM'] > 15]['WidthM'].median())

In [ ]:
# Make dict for legend labels
flags = gdf_PIXC.classification.unique()
codes = {idx:k for idx, k in enumerate(flags, start=2)}

# Set palette
palette = {2: 'darkolivegreen', 3: 'cornflowerblue', 4: 'blue', 5: 'hotpink',
           6: 'darkkhaki', 7: 'springgreen'}

# Print min/max indices in slant range for SLC plots
print(f'Azimuth: {gdf_PIXC_clip.azimuth_index.min()}, {gdf_PIXC_clip.azimuth_index.max()}')
print(f'Range: {gdf_PIXC_clip.range_index.min()}, {gdf_PIXC_clip.range_index.max()}')

In [ ]:
nums = list(np.unique(gdf_PIXC_clip.classification).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,7))
buffer_clip.plot(ax=ax, alpha=0.5, color='k')
gdf_PIXC_clip.plot(column='classification', cmap=cmap, ax=ax, s=0.5, alpha=0.2)
gdf_PIXC_buffer.plot(column='classification', cmap=cmap, ax=ax, s=0.5)

ctx.add_basemap(ax,
                crs=gdf_PIXC_clip.crs,
                source=ctx.providers.CartoDB.Positron,
                # zoom=12,
                attribution='')

ax.text(0.99, 0.01, ctx.providers.CartoDB.Positron.attribution, 
        transform=ax.transAxes, 
        ha='right', va='bottom', 
        fontsize=7, 
        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2, foreground="w")])

plt.xlim(bounds[0], bounds[2]);
plt.ylim(bounds[1], bounds[3]);

ax.axis('off')
ax.set_position([0, 0, 1, 1]) 

# plt.savefig(fname=save_path + 'pixc_virgin.png', dpi=300)

### Tolt River

In [ ]:
granule = '_045_345_239R_'
epsg_code = '32610' # WGS84 / UTM zone 10N
huc4 = '1711'
# bounds = [583000, 5276000, 588000, 5283000]
bounds = [583500, 5277000, 587500, 5282000]

In [ ]:
gdf_PIXC = read_and_prep_pixc(granule=granule, epsg_code=epsg_code)

# Read in the buffered reaches (for plotting)
# and the buffered reaches with a bit extra (for clipping pixels to show corridor)
buffer = read_and_prep_nhd_buffer(huc4=huc4, epsg_code=epsg_code, extra=False)
buffer_extra = read_and_prep_nhd_buffer(huc4=huc4, epsg_code=epsg_code, extra=True)

# Clip to bounds
gdf_PIXC_clip = gdf_PIXC.clip(mask=bounds)
gdf_PIXC_buffer = gdf_PIXC_clip.sjoin(buffer_extra)
buffer_clip = buffer.clip(mask=bounds)

print(buffer_clip[buffer_clip['WidthM'] > 15]['WidthM'].median())

In [ ]:
# Make dict for legend labels
flags = gdf_PIXC.classification.unique()
codes = {idx:k for idx, k in enumerate(flags, start=2)}

# Set palette
palette = {2: 'darkolivegreen', 3: 'cornflowerblue', 4: 'blue', 5: 'hotpink',
           6: 'darkkhaki', 7: 'springgreen'}

# Print min/max indices in slant range for SLC plots
print(f'Azimuth: {gdf_PIXC_clip.azimuth_index.min()}, {gdf_PIXC_clip.azimuth_index.max()}')
print(f'Range: {gdf_PIXC_clip.range_index.min()}, {gdf_PIXC_clip.range_index.max()}')

In [ ]:
nums = list(np.unique(gdf_PIXC_clip.classification).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,7))
buffer_clip.plot(ax=ax, alpha=0.5, color='k')
gdf_PIXC_clip.plot(column='classification', cmap=cmap, ax=ax, s=0.5, alpha=0.2)
gdf_PIXC_buffer.plot(column='classification', cmap=cmap, ax=ax, s=0.5)

ctx.add_basemap(ax,
                crs=gdf_PIXC_clip.crs,
                source=ctx.providers.CartoDB.Positron,
                # zoom=12,
                attribution='')

ax.text(0.99, 0.01, ctx.providers.CartoDB.Positron.attribution, 
        transform=ax.transAxes, 
        ha='right', va='bottom', 
        fontsize=7,
        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2, foreground="w")])

plt.xlim(bounds[0], bounds[2]);
plt.ylim(bounds[1], bounds[3]);

ax.axis('off')
ax.set_position([0, 0, 1, 1]) 

plt.savefig(fname=save_path + 'pixc_tolt.png', dpi=300)

In [ ]:
square_bounds = [583500, 5277500, 587500, 5281500]

# PLOT SQUARE VERSION
nums = list(np.unique(gdf_PIXC_clip.classification).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,5))
buffer_clip.plot(ax=ax, alpha=0.5, color='k')
gdf_PIXC_clip.plot(column='classification', cmap=cmap, ax=ax, s=0.5, alpha=0.2)
gdf_PIXC_buffer.plot(column='classification', cmap=cmap, ax=ax, s=0.5)

ctx.add_basemap(ax,
                crs=gdf_PIXC_clip.crs,
                source=ctx.providers.CartoDB.Positron,
                # zoom=12,
                attribution='')

ax.text(0.99, 0.01, ctx.providers.CartoDB.Positron.attribution, 
        transform=ax.transAxes, 
        ha='right', va='bottom', 
        fontsize=7,
        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2, foreground="w")])

plt.xlim(square_bounds[0], square_bounds[2]);
plt.ylim(square_bounds[1], square_bounds[3]);

ax.axis('off')
ax.set_position([0, 0, 1, 1]) 

plt.savefig(fname=save_path + 'pixc_tolt_square.png', dpi=300)

### Fort River

In [ ]:
granule = '_032_298_080L_'
epsg_code = '32618' # WGS84 / UTM zone 18N
huc4 = '0108'
# bounds = [698000, 4687500, 703000, 4694500]
bounds = [698950, 4688900, 701950, 4691900]

In [ ]:
gdf_PIXC = read_and_prep_pixc(granule=granule, epsg_code=epsg_code)

# Read in the buffered reaches (for plotting)
# and the buffered reaches with a bit extra (for clipping pixels to show corridor)
buffer = read_and_prep_nhd_buffer(huc4=huc4, epsg_code=epsg_code, extra=False)
buffer_extra = read_and_prep_nhd_buffer(huc4=huc4, epsg_code=epsg_code, extra=True)

# Clip to bounds
gdf_PIXC_clip = gdf_PIXC.clip(mask=bounds)
gdf_PIXC_buffer = gdf_PIXC_clip.sjoin(buffer_extra)
buffer_clip = buffer.clip(mask=bounds)

print(buffer_clip[(buffer_clip['WidthM'] > 15) & (buffer_clip['GNIS_Name'] == 'Fort River')]['WidthM'].median())

In [ ]:
# Make dict for legend labels
flags = gdf_PIXC.classification.unique()
codes = {idx:k for idx, k in enumerate(flags, start=2)}

# Set palette
palette = {2: 'darkolivegreen', 3: 'cornflowerblue', 4: 'blue', 5: 'hotpink',
           6: 'darkkhaki', 7: 'springgreen'}

# Print min/max indices in slant range for SLC plots
print(f'Azimuth: {gdf_PIXC_clip.azimuth_index.min()}, {gdf_PIXC_clip.azimuth_index.max()}')
print(f'Range: {gdf_PIXC_clip.range_index.min()}, {gdf_PIXC_clip.range_index.max()}')

In [ ]:
nums = list(np.unique(gdf_PIXC_clip.classification).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,7))
buffer_clip.plot(ax=ax, alpha=0.5, color='k')
gdf_PIXC_clip.plot(column='classification', cmap=cmap, ax=ax, s=0.5, alpha=0.2)
gdf_PIXC_buffer.plot(column='classification', cmap=cmap, ax=ax, s=0.5)

ctx.add_basemap(ax,
                crs=gdf_PIXC_clip.crs,
                source=ctx.providers.CartoDB.Positron,
                # zoom=12,
                attribution='')

ax.text(0.99, 0.01, ctx.providers.CartoDB.Positron.attribution, 
        transform=ax.transAxes, 
        ha='right', va='bottom', 
        fontsize=7, 
        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2, foreground="w")])

plt.xlim(bounds[0], bounds[2]);
plt.ylim(bounds[1], bounds[3]);

ax.axis('off')
ax.set_position([0, 0, 1, 1]) 

# plt.savefig(fname=save_path + 'pixc_fort.png', dpi=300)